# 四层探测器缪子散射成像：PoCA 基线 Notebook

这是本项目的**学习版分析 Notebook**。它从 Geant4 导出的 `muon_tracks.csv` 出发，依次完成：

1. 检查四层探测器击中数据；
2. 理解散射角与 PoCA 点；
3. 将 PoCA 点累积成固定体素图像；
4. 在不同事件数下计算 IoU 和定位误差。

建议严格从上到下逐格运行。每个代码单元前都说明了“为什么需要这一步”。

## 0. 这份数据在物理上表示什么

Geant4 中有四个平面探测器：上游两层用于拟合**入射轨迹**，下游两层用于拟合**出射轨迹**。缪子穿过 U 型铝目标后会发生多重库仑散射，因此两条直线会有一个小夹角。

PoCA（Point of Closest Approach）把两条直线最近点连线的中点，近似为该缪子主要发生散射的位置。这个近似并不完美，但简单、可解释，适合做第一条基线。

In [ ]:
# 导入本 Notebook 需要的库。
# pandas：读取和整理 CSV；numpy：数值计算；matplotlib：绘图。
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让图中的中文在大多数 macOS 环境里更容易显示。
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# Notebook 位于 muon_analysis 文件夹，因此 CSV 默认就在同一目录。
DATA_PATH = Path('muon_tracks.csv')
print('准备读取：', DATA_PATH.resolve())

## 1. 读取 CSV，并先确认数据是否完整

这一格不做任何成像。先查看列名和前几行，确认 Geant4 输出中确实包含四层击中、PoCA 坐标和散射角。这样若后面报错，能更快判断是数据问题还是算法问题。

In [ ]:
# 读取 Geant4 输出。每一行对应一条通过四层探测器的完整缪子径迹。
df = pd.read_csv(DATA_PATH)

print(f'完整径迹数：{len(df):,}')
print('\nCSV 字段：')
print(df.columns.tolist())
display(df.head())

### 字段速查

- `up1_*`、`up2_*`：上游两层的击中坐标；
- `down1_*`、`down2_*`：下游两层的击中坐标；
- `poca_*`：在 C++ 端由两条拟合直线计算的 PoCA 坐标；
- `theta_x_mrad`、`theta_y_mrad`：x-z 和 y-z 平面上的投影散射角；
- `theta_mrad`：总散射角，后面用它的平方作为体素权重。

坐标单位是 mm，角度单位是 mrad。

In [ ]:
# 如果下面缺少任何字段，说明当前 CSV 不是本项目需要的完整输出格式。
REQUIRED_COLUMNS = {
    'up1_x_mm', 'up1_y_mm', 'up1_z_mm',
    'up2_x_mm', 'up2_y_mm', 'up2_z_mm',
    'down1_x_mm', 'down1_y_mm', 'down1_z_mm',
    'down2_x_mm', 'down2_y_mm', 'down2_z_mm',
    'poca_x_mm', 'poca_y_mm', 'poca_z_mm', 'theta_mrad',
}
missing = REQUIRED_COLUMNS - set(df.columns)
assert not missing, f'CSV 缺少字段：{sorted(missing)}'
print('✓ CSV 字段检查通过。')

## 2. 看一眼上下游击中位置

理想入射模式下，探测器平面上的击中分布接近均匀；这张图本身不直接显示 U 型目标。真正携带目标信息的是上下游轨迹之间的变化，即散射角和 PoCA 点。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

# 上游：缪子进入目标前的两次位置测量。
axes[0].scatter(df['up1_x_mm'], df['up1_y_mm'], s=3, alpha=0.35, label='上游第 1 层')
axes[0].scatter(df['up2_x_mm'], df['up2_y_mm'], s=3, alpha=0.35, label='上游第 2 层')
axes[0].set(title='入射侧探测器击中', xlabel='x [mm]', ylabel='y [mm]')
axes[0].set_aspect('equal')
axes[0].legend()

# 下游：缪子穿过目标后的两次位置测量。
axes[1].scatter(df['down1_x_mm'], df['down1_y_mm'], s=3, alpha=0.35, label='下游第 1 层')
axes[1].scatter(df['down2_x_mm'], df['down2_y_mm'], s=3, alpha=0.35, label='下游第 2 层')
axes[1].set(title='出射侧探测器击中', xlabel='x [mm]', ylabel='y [mm]')
axes[1].set_aspect('equal')
axes[1].legend()
plt.show()

## 3. 散射角分布

`theta_mrad` 越大，通常意味着缪子在目标附近经历了更明显的散射。后续固定体素 PoCA 不只是计数，而是累加 $\theta^2$，从而使大散射事件对图像贡献更大。

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.hist(df['theta_mrad'], bins=60, histtype='stepfilled', alpha=0.55, color='#d95f02')
plt.xlabel('总散射角 theta [mrad]')
plt.ylabel('事件数')
plt.title('缪子散射角分布')
plt.grid(alpha=0.25)
plt.show()

print(df['theta_mrad'].describe())

## 4. 先观察 PoCA 三维点云

颜色表示散射角。三维点云适合帮助你建立直觉，但论文中通常还需要体素图，因为体素图可以稳定地计算轮廓、IoU 和定位误差。

In [ ]:
fig = plt.figure(figsize=(8, 6.5))
ax = fig.add_subplot(111, projection='3d')

# 使用少量点可以让三维图更清晰；若数据更少则直接使用全部。
plot_df = df.sample(min(5000, len(df)), random_state=20260921)
scatter = ax.scatter(
    plot_df['poca_x_mm'], plot_df['poca_y_mm'], plot_df['poca_z_mm'],
    c=plot_df['theta_mrad'], s=4, alpha=0.6, cmap='inferno'
)
ax.set(xlabel='PoCA x [mm]', ylabel='PoCA y [mm]', zlabel='PoCA z [mm]',
       title='PoCA 三维散射点（颜色：总散射角）')
fig.colorbar(scatter, ax=ax, shrink=0.7, label='theta [mrad]')
plt.show()

## 5. 设定固定体素 PoCA 的参数

这些参数就是你之后做改进算法时必须固定的**基线条件**。先不要随意改动；改参数应该单独成为一组实验，并记录到论文和 Git 提交中。

In [ ]:
# 成像区域覆盖 U 型目标（目标在 xy 平面约为 ±60 mm）。
XY_LIMIT_MM = 90.0
Z_LIMIT_MM = 100.0

# 32³ 是当前论文基线网格。增大 BINS 会得到更细图像，
# 但在低统计量下也会让每个体素更稀疏、更噪。
BINS = 32

# 体素权重：每个体素累加 theta_mrad²。
WEIGHT_DESCRIPTION = 'sum(theta_mrad²)'

print(f'成像范围：x/y ±{XY_LIMIT_MM:g} mm，z ±{Z_LIMIT_MM:g} mm')
print(f'体素网格：{BINS} × {BINS} × {BINS}')
print(f'体素权重：{WEIGHT_DESCRIPTION}')

## 6. 将 PoCA 点累积到三维体素

函数做三件事：

1. 去掉成像范围外的 PoCA 点；
2. 用 `numpy.histogramdd` 把三维空间分成规则体素；
3. 在每个体素中累加 $\theta^2$。

注意：函数不修改原始 `df`，因此每次实验都可以从同一份 Geant4 数据公平比较。

In [ ]:
def voxelize_poca(data: pd.DataFrame, bins: int = BINS,
                  xy_limit: float = XY_LIMIT_MM,
                  z_limit: float = Z_LIMIT_MM):
    """把 PoCA 事件变成固定网格的三维散射强度。

    参数
    ----
    data : 需要重建的事件子集。
    bins : x、y、z 三个方向使用的体素数。
    xy_limit, z_limit : 成像范围半宽，单位 mm。

    返回
    ----
    volume : shape=(bins, bins, bins) 的体素强度数组。
    edges : 三个方向的体素边界。
    valid : 实际落入成像范围的事件。它用于统计，而非修改输入数据。
    """
    # 仅保留落入重建区域的 PoCA 点；范围外的点不参与体素累积。
    valid = data[
        (data['poca_x_mm'].abs() <= xy_limit)
        & (data['poca_y_mm'].abs() <= xy_limit)
        & (data['poca_z_mm'].abs() <= z_limit)
    ].copy()

    # histogramdd 需要每个方向的分箱边界，而不是体素中心。
    edges = [
        np.linspace(-xy_limit, xy_limit, bins + 1),
        np.linspace(-xy_limit, xy_limit, bins + 1),
        np.linspace(-z_limit, z_limit, bins + 1),
    ]

    # theta² 让高散射事件比近似直行事件拥有更大权重。
    weights = valid['theta_mrad'].to_numpy() ** 2
    volume, edges = np.histogramdd(
        valid[['poca_x_mm', 'poca_y_mm', 'poca_z_mm']].to_numpy(),
        bins=edges,
        weights=weights,
    )
    return volume, edges, valid


volume, voxel_edges, valid_events = voxelize_poca(df)
print(f'进入成像范围的 PoCA 点：{len(valid_events):,} / {len(df):,}')
print('体素数组形状：', volume.shape)

## 7. 得到最基础的 U 型目标 XY 图像

这里沿 z 方向把三维体素强度相加，得到 xy 投影。你应该能看到开口朝 +y 的 U 型大致轮廓。若看不到，先检查：是否运行了 Geant4、是否使用正确 CSV、成像范围是否仍覆盖目标。

In [ ]:
# axis=2 对应 z 方向；求和后留下 x-y 平面强度图。
xy_image = volume.sum(axis=2)

plt.figure(figsize=(7, 6))
mesh = plt.pcolormesh(
    voxel_edges[0], voxel_edges[1], xy_image.T,
    shading='auto', cmap='inferno'
)
plt.colorbar(mesh, label=r'$\sum \theta^2$ [mrad$^2$]')
plt.xlabel('PoCA x [mm]')
plt.ylabel('PoCA y [mm]')
plt.title('固定体素 PoCA：U 型目标 xy 投影')
plt.axis('equal')
plt.show()

## 8. 构造 U 型真值掩膜，并定义评价指标

IoU 需要真实目标轮廓。这里的真值掩膜严格对应 Geant4 中的 U 型铝目标：外尺寸 120 mm × 120 mm、壁厚 30 mm、开口朝 +y。

为了不让不同事件数各自“挑选最有利阈值”，采用**等面积 Top-k**：预测图中取强度最高的 k 个像素，且 k 固定为真值掩膜的像素数。

In [ ]:
def u_target_mask(x_centers: np.ndarray, y_centers: np.ndarray) -> np.ndarray:
    """返回与 DetectorConstruction.cc 一致的 U 型二维真值掩膜。"""
    x_grid, y_grid = np.meshgrid(x_centers, y_centers, indexing='ij')

    # U 的底边：y 从 -60 到 -30 mm。
    bottom = (np.abs(x_grid) <= 60.0) & (y_grid >= -60.0) & (y_grid <= -30.0)
    # 左、右竖边：x 分别位于 [-60, -30] 与 [30, 60] mm。
    left = (x_grid >= -60.0) & (x_grid <= -30.0) & (y_grid >= -30.0) & (y_grid <= 60.0)
    right = (x_grid >= 30.0) & (x_grid <= 60.0) & (y_grid >= -30.0) & (y_grid <= 60.0)
    return bottom | left | right


def evaluate_xy_image(image: np.ndarray, x_centers: np.ndarray, y_centers: np.ndarray):
    """计算等面积 IoU 和散射强度质心定位误差。"""
    truth = u_target_mask(x_centers, y_centers)
    k = int(truth.sum())

    # 强度最高的 k 个体素像素视为重建得到的目标区域。
    predicted = np.zeros_like(truth, dtype=bool)
    top_indices = np.argpartition(image.ravel(), -k)[-k:]
    predicted.ravel()[top_indices] = True

    intersection = np.logical_and(predicted, truth).sum()
    union = np.logical_or(predicted, truth).sum()
    iou = intersection / union if union else np.nan

    # 强度加权质心与真值掩膜几何中心的距离定义为定位误差。
    x_grid, y_grid = np.meshgrid(x_centers, y_centers, indexing='ij')
    total = image.sum()
    rec_x = (image * x_grid).sum() / total
    rec_y = (image * y_grid).sum() / total
    true_x, true_y = x_grid[truth].mean(), y_grid[truth].mean()
    localization_error_mm = np.hypot(rec_x - true_x, rec_y - true_y)
    return iou, localization_error_mm, truth, predicted


x_centers = (voxel_edges[0][:-1] + voxel_edges[0][1:]) / 2
y_centers = (voxel_edges[1][:-1] + voxel_edges[1][1:]) / 2
iou, loc_error, truth_mask, predicted_mask = evaluate_xy_image(xy_image, x_centers, y_centers)
print(f'全部事件基线 IoU：{iou:.4f}')
print(f'全部事件定位误差：{loc_error:.4f} mm')

## 9. 低统计量：只改变事件数，其他条件保持不变

这一步是论文的基线实验。每个事件数重复随机抽样多次，报告均值和标准差，避免一次抽样的偶然性被误认为算法效果。这里使用与已完成基线实验一致的随机种子和事件数。

In [ ]:
EVENT_COUNTS = [1000, 3000, 5000]
REPEATS = 5
RANDOM_SEED = 20260921

rng = np.random.default_rng(RANDOM_SEED)
records = []

for n_events in EVENT_COUNTS:
    for repeat in range(REPEATS):
        # 无放回抽样：同一次重建中同一条缪子事件不会被重复使用。
        subset = df.iloc[rng.choice(len(df), size=n_events, replace=False)]
        sub_volume, sub_edges, sub_valid = voxelize_poca(subset)
        sub_image = sub_volume.sum(axis=2)
        sub_iou, sub_error, _, _ = evaluate_xy_image(sub_image, x_centers, y_centers)
        records.append({
            'event_count': n_events,
            'repeat': repeat + 1,
            'poca_points_in_volume': len(sub_valid),
            'iou': sub_iou,
            'localization_error_mm': sub_error,
        })

results = pd.DataFrame(records)
summary = results.groupby('event_count').agg(
    repeats=('repeat', 'count'),
    iou_mean=('iou', 'mean'), iou_std=('iou', 'std'),
    error_mean=('localization_error_mm', 'mean'),
    error_std=('localization_error_mm', 'std'),
).reset_index()
display(summary)

## 10. 画出事件数与图像质量的关系

理想情况下，事件数增加时 IoU 上升、定位误差下降。若你在改动算法后重新运行这部分，就能和固定体素 PoCA 基线进行公平对比。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].errorbar(summary['event_count'], summary['iou_mean'], yerr=summary['iou_std'],
                 marker='o', capsize=4, color='#1f77b4')
axes[0].set_xscale('log')
axes[0].set(xlabel='输入事件数（对数坐标）', ylabel='等面积 IoU', title='事件数与轮廓重叠度')
axes[0].grid(alpha=0.25)

axes[1].errorbar(summary['event_count'], summary['error_mean'], yerr=summary['error_std'],
                 marker='o', capsize=4, color='#d62728')
axes[1].set_xscale('log')
axes[1].set(xlabel='输入事件数（对数坐标）', ylabel='定位误差 [mm]', title='事件数与定位误差')
axes[1].grid(alpha=0.25)
plt.show()

## 11. 下一步要做什么

你现在已经完成了传统固定体素 PoCA 的学习和复现实验。下一步是**散射角筛选 POCA**：

1. 在重建前加一条 `theta_mrad >= threshold` 的筛选；
2. 扫描多个阈值，而不是凭图像主观选择；
3. 对每个阈值重复本 Notebook 的低统计量实验；
4. 与这里的 IoU、定位误差和运行时间比较；
5. 将最优参数、随机种子、图和表写进论文第 6、7 章，并提交 GitHub。

**重要原则：** 每次只改变一个核心变量。否则即使图像变好，也无法说明真正起作用的是哪一个改动。